In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import gc
from utils.metrics1 import cacluate_interval_score, evaluate_regress
from utils.tools import converge, set_seed
from utils.build_metric_summary_df import build_metric_summary_df
from data.data_process import DataLoader
from data.data_process_external import DataLoaderExternal
from data.time_series_process import build_seq_samples,build_nonoverlap_target,flatten_pred_matrix,fit_seq_norm,apply_seq_norm
from models.custom_lgbm import CustomLightgbm
from models.custom_lstmr import CustomLSTM1
from models.custom_mlp import CustomMLP
from models.custom_timelstm import CustomLSTMTIME1
from models.custom_timetransformer import CustomTransformerTIME1


In [5]:
set_seed(3407)

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

day_set = 30

zhixin = np.array([0, 0.3, 0.5, 0.7, 0.9], dtype=float)
zhixin1 = zhixin[1:]

data_set_list = ['data_NG']

site_config = {
    'data_FBS': {
        'std_meridian_deg': 120,
        'longitude': 104.0,
        'latitude': 37.0
    },
    'data_JS': {
        'std_meridian_deg': 120,
        'longitude': 93.4,
        'latitude': 36.0
    },
    'data_NG': {
        'std_meridian_deg': 120,
        'longitude': 113.4,
        'latitude': 38.0
    },
    'data_GED': {
        'std_meridian_deg': 0,
        'longitude': 133.87,
        'latitude': -23.7
    },
    'data_AU': {
        'std_meridian_deg': 120,
        'longitude': 130.98,
        'latitude': -25.24
    }
}


# method_choose_list = ['MLP', 'LSTM', 'LSTMTIME','TransformerTIME']
method_choose_list = ['TransformerTIME','LSTMTIME', 'LSTM','MLP']
method_choose_list1 = []

future_horizons_num = 96
past_lags_num = 96

index = np.arange(0, len(method_choose_list))

prdict_y_list_all = []
true_y_list_all = []
lower_y_list_all = []
up_y_list_all = []

output_df_ngboost_all = []
point_predict_all = []
interval_predict_all = []
df_test_list_all = []

prdict_y_list_external_all = []
true_y_list_external_all = []
lower_y_list_external_all = []
up_y_list_external_all = []
point_predict_external_all = []
interval_predict_external_all = []
df_test_list_external_all = []


for index_set in index:
    method_choose = method_choose_list[index_set]
    method_choose_list1.append(method_choose)

    station_name_list = []
    point_predict = []
    interval_predict = []

    true_y_list = []
    prdict_y_list = []
    lower_y_list = []
    up_y_list = []

    true_y_list_external = []
    prdict_y_list_external = []
    lower_y_list_external = []
    up_y_list_external = []
    point_predict_external = []
    interval_predict_external = []

    output_df = None

    for data_test_name in data_set_list:
        project_root = Path.cwd()
        dataset_folder_path = project_root / 'datasets' / data_test_name
        print(dataset_folder_path)

        dataset_paths = sorted([
            file_name for file_name in os.listdir(dataset_folder_path)
            if file_name.endswith('.xlsx') or file_name.endswith('.xls')
        ])

        std_meridian_deg = site_config[data_test_name]['std_meridian_deg']
        longitude = site_config[data_test_name]['longitude']
        latitude = site_config[data_test_name]['latitude']

#         for station_idx in range(0, len(dataset_paths)):
        for station_idx in range(0, 2):
            dataset_path = dataset_paths[station_idx]
            station_name = dataset_path[:-5]

            print(f'Processing {method_choose} - {data_test_name} - {station_name}')

            df_all = pd.read_excel(dataset_folder_path / dataset_path)
            df_process = df_all.dropna().reset_index(drop=True)

            if len(df_process) < int(day_set * 1.1 * 96):
                print(f'[Warning] {station_name} Insufficient sample length. Skipped.')
                continue

            station_name_list.append(station_name)

            df_process_time = df_process.copy()
            df_process['row_index'] = np.arange(len(df_process))

            train_end = int(day_set * 0.8 * 96)
            valid_end = int(day_set * 0.9 * 96)
            test_end = int(day_set * 1.0 * 96)
            ext_end = int(day_set * 1.1 * 96)

            # 
            train_end = (train_end // 96) * 96
            valid_end = (valid_end // 96) * 96
            test_end = (test_end // 96) * 96
            ext_end = (ext_end // 96) * 96

            target_col = df_process_time.columns[-1]
            feature_cols = list(df_process_time.columns[1:-1])

            # =======================================
            # A. tabular task
            # =======================================
            if method_choose in ['LightGBM', 'MLP', 'LSTM']:
                df_train_list = df_process[(df_process['row_index'] >= 0) & (df_process['row_index'] < train_end)]
                df_vaild_list = df_process[(df_process['row_index'] >= train_end) & (df_process['row_index'] < valid_end)]
                df_test_list = df_process[(df_process['row_index'] >= valid_end) & (df_process['row_index'] < test_end)]
                df_test_list_external = df_process[(df_process['row_index'] >= test_end) & (df_process['row_index'] < ext_end)]

                drop_column = ['row_index']

                df_train_list1 = df_train_list.drop(drop_column, axis=1).copy()
                df_vaild_list1 = df_vaild_list.drop(drop_column, axis=1).copy()
                df_test_list1 = df_test_list.drop(drop_column, axis=1).copy()
                df_test_list_external1 = df_test_list_external.drop(drop_column, axis=1).copy()

                dataloader = DataLoader(
                    latitude, longitude, std_meridian_deg,
                    df_train_list1, df_vaild_list1, df_test_list1)

                (
                    train_x, train_y, train_x_nor, train_y_nor,
                    train_aug_x, train_aug_y, train_aug_x_nor, train_aug_y_nor,
                    val_x, val_y, val_x_nor, val_y_nor,
                    val_aug_x, val_aug_y, val_aug_x_nor, val_aug_y_nor,
                    test_x, test_y, test_x_nor, test_y_nor,
                    test_aug_x, test_aug_y, test_aug_x_nor, test_aug_y_nor,
                    y_mean, y_std
                ) = dataloader.get_dataset()

                train_aug_x_nor=train_x_nor.copy()
                val_aug_x_nor=val_x_nor.copy()
                test_aug_x_nor=test_x_nor.copy()
                # test_aug_x_nor=test_x_nor.copy()


                train_aug_x_mean = train_aug_x.mean(0)
                train_aug_x_std = train_aug_x.std(0)
                train_aug_x_std[train_aug_x_std == 0] = 1.0

                dataloader_external = DataLoaderExternal(
                    latitude=latitude,
                    longitude=longitude,
                    std_meridian_deg=std_meridian_deg,
                    external_df=df_test_list_external1,
                    aug_x_mean=train_aug_x_mean,
                    aug_x_std=train_aug_x_std,
                    has_target=True
                )
                test_aug_x_ext_nor1, test_y_external = dataloader_external.get_dataset1()
                test_aug_x_ext_nor=test_aug_x_ext_nor1[:,:-4]
                if method_choose == 'LightGBM':
                    lgbm_model = CustomLightgbm(zhixin)
                    lgbm_model.train(train_aug_x_nor, train_y_nor, val_aug_x_nor, val_y_nor)
                    y_middle, upper, lower = lgbm_model.predict(test_aug_x_nor, y_mean, y_std)
                    y_middle_external, upper_external, lower_external = lgbm_model.predict(test_aug_x_ext_nor, y_mean, y_std)

                elif method_choose == 'LSTM':
                    gc.collect()
                    torch.cuda.empty_cache()
                    lstm_model = CustomLSTM1(
                        zhixin,
                        hidden_dim=64,
                        num_layers=1,
                        lr=0.001,
                        batch_size=256,
                        num_epochs=60,
                        method_choose='LSTM'
                    )
                    lstm_model.train(train_aug_x_nor, train_y_nor, val_aug_x_nor, val_y_nor)
                    y_middle, upper, lower = lstm_model.predict(test_aug_x_nor, y_mean, y_std)
                    y_middle_external, upper_external, lower_external = lstm_model.predict(test_aug_x_ext_nor, y_mean, y_std)

                elif method_choose == 'MLP':
                    gc.collect()
                    torch.cuda.empty_cache()
                    mlp_model = CustomMLP(
                        zhixin,
                        input_dim=train_aug_x_nor.shape[1],
                        hidden_dim=32,
                        lr=0.001,
                        batch_size=256,
                        num_epochs=60,
                        method_choose='MLP'
                    )
                    mlp_model.train(train_aug_x_nor, train_y_nor, val_aug_x_nor, val_y_nor)
                    y_middle, upper, lower = mlp_model.predict(test_aug_x_nor, y_mean, y_std)
                    y_middle_external, upper_external, lower_external = mlp_model.predict(test_aug_x_ext_nor, y_mean, y_std)

                y_true_eval = np.asarray(test_y).reshape(-1)
                y_true_ext_eval = np.asarray(test_y_external).reshape(-1)
                y_true_eval1=y_true_eval.copy()
                y_true_ext_eval1=y_true_eval.copy()

            # =======================================
            # B. TransformerTIME：timeseries_task
            # =======================================
            elif method_choose == 'TransformerTIME':
                # train set stride=1 ;/test/val stride=96
                x_train_past, x_train_future, y_train_matrix, train_origins = build_seq_samples(
                    df=df_process_time,
                    feature_cols=feature_cols,
                    target_col=target_col,
                    past_lags=past_lags_num,
                    horizon=future_horizons_num,
                    origin_start=0,
                    origin_end=train_end,
                    stride=1
                )

                x_valid_past, x_valid_future, y_valid_matrix, valid_origins = build_seq_samples(
                    df=df_process_time,
                    feature_cols=feature_cols,
                    target_col=target_col,
                    past_lags=past_lags_num,
                    horizon=future_horizons_num,
                    origin_start=train_end,
                    origin_end=valid_end,
                    stride=96
                )

                x_test_past, x_test_future, y_test_matrix, test_origins = build_seq_samples(
                    df=df_process_time,
                    feature_cols=feature_cols,
                    target_col=target_col,
                    past_lags=past_lags_num,
                    horizon=future_horizons_num,
                    origin_start=valid_end,
                    origin_end=test_end,
                    stride=96
                )

                x_ext_past, x_ext_future, y_ext_matrix, ext_origins = build_seq_samples(
                    df=df_process_time,
                    feature_cols=feature_cols,
                    target_col=target_col,
                    past_lags=past_lags_num,
                    horizon=future_horizons_num,
                    origin_start=test_end,
                    origin_end=ext_end,
                    stride=96
                )

                if x_test_past.shape[0] == 0 or x_ext_past.shape[0] == 0:
                    print(f'[Warning] {station_name} LSTMTIME wwindows')
                    continue

                norm_dict = fit_seq_norm(x_train_past, x_train_future, y_train_matrix)

                train_x_past_nor, train_x_future_nor, train_y_nor = apply_seq_norm(
                    x_train_past, x_train_future, y_train_matrix, norm_dict)
                valid_x_past_nor, valid_x_future_nor, valid_y_nor = apply_seq_norm(
                    x_valid_past, x_valid_future, y_valid_matrix, norm_dict)
                test_x_past_nor, test_x_future_nor, _ = apply_seq_norm(
                    x_test_past, x_test_future, None, norm_dict)
                ext_x_past_nor, ext_x_future_nor, _ = apply_seq_norm(
                    x_ext_past, x_ext_future, None, norm_dict)

                gc.collect()
                torch.cuda.empty_cache()

                seq_model = CustomTransformerTIME1(
                    zhixin=zhixin,
                    past_input_dim=train_x_past_nor.shape[-1],
                    future_input_dim=train_x_future_nor.shape[-1],
                    horizon=96,
                    hidden_dim=64,
                    num_heads=4,
                    num_layers=2,
                    ff_dim=64,
                    lr=0.001,
                    batch_size=128,
                    num_epochs=60,
                    dropout=0.1,
                    patience=10,
                    pool_mode="last"
                )


                seq_model.train(
                    train_x_past_nor, train_x_future_nor, train_y_nor,
                    valid_x_past_nor, valid_x_future_nor, valid_y_nor
                )

                y_middle_matrix, upper_matrix, lower_matrix = seq_model.predict(
                    test_x_past_nor, test_x_future_nor,
                    norm_dict['y_mean'], norm_dict['y_std']
                )
                y_middle_external_matrix, upper_external_matrix, lower_external_matrix = seq_model.predict(
                    ext_x_past_nor, ext_x_future_nor,
                    norm_dict['y_mean'], norm_dict['y_std']
                )

                
                y_middle, upper, lower = flatten_pred_matrix(y_middle_matrix, upper_matrix, lower_matrix)
                y_middle_external, upper_external, lower_external = flatten_pred_matrix(
                    y_middle_external_matrix, upper_external_matrix, lower_external_matrix)

                y_true_eval = y_test_matrix.reshape(-1)
                y_true_ext_eval = y_ext_matrix.reshape(-1)

                # # Verify whether the test intervals used in the sequential forecasting task are exactly aligned
                # with those used in the original tabular forecasting task
                
                raw_test_target = build_nonoverlap_target(df_process_time, target_col, valid_end, test_end)
                raw_ext_target = build_nonoverlap_target(df_process_time, target_col, test_end, ext_end)

                if len(raw_test_target) >= len(y_true_eval):
                    assert np.allclose(raw_test_target[:len(y_true_eval)], y_true_eval), 'The test set of LSTMTIME is not aligned with the original test interval.'
                if len(raw_ext_target) >= len(y_true_ext_eval):
                    assert np.allclose(raw_ext_target[:len(y_true_ext_eval)], y_true_ext_eval), 'The external test set of LSTMTIME is not aligned with the original external test interval.'

                # 
                df_test_list = df_process.iloc[valid_end:test_end].copy()
                df_test_list_external = df_process.iloc[test_end:ext_end].copy()

                print(f'TransformerTIME test matrix shape: {y_middle_matrix.shape}')
                print(f'TransformerTIME external matrix shape: {y_middle_external_matrix.shape}')

            elif method_choose == 'LSTMTIME':
                # train set stride=1 ;/test/val stride=96
                x_train_past, x_train_future, y_train_matrix, train_origins = build_seq_samples(
                    df=df_process_time,
                    feature_cols=feature_cols,
                    target_col=target_col,
                    past_lags=past_lags_num,
                    horizon=future_horizons_num,
                    origin_start=0,
                    origin_end=train_end,
                    stride=1
                )

                x_valid_past, x_valid_future, y_valid_matrix, valid_origins = build_seq_samples(
                    df=df_process_time,
                    feature_cols=feature_cols,
                    target_col=target_col,
                    past_lags=past_lags_num,
                    horizon=future_horizons_num,
                    origin_start=train_end,
                    origin_end=valid_end,
                    stride=96
                )

                x_test_past, x_test_future, y_test_matrix, test_origins = build_seq_samples(
                    df=df_process_time,
                    feature_cols=feature_cols,
                    target_col=target_col,
                    past_lags=past_lags_num,
                    horizon=future_horizons_num,
                    origin_start=valid_end,
                    origin_end=test_end,
                    stride=96
                )

                x_ext_past, x_ext_future, y_ext_matrix, ext_origins = build_seq_samples(
                    df=df_process_time,
                    feature_cols=feature_cols,
                    target_col=target_col,
                    past_lags=past_lags_num,
                    horizon=future_horizons_num,
                    origin_start=test_end,
                    origin_end=ext_end,
                    stride=96
                )

                if x_test_past.shape[0] == 0 or x_ext_past.shape[0] == 0:
                    print(f'[Warning] {station_name} LSTMTIME wwindows')
                    continue

                norm_dict = fit_seq_norm(x_train_past, x_train_future, y_train_matrix)

                train_x_past_nor, train_x_future_nor, train_y_nor = apply_seq_norm(
                    x_train_past, x_train_future, y_train_matrix, norm_dict)
                valid_x_past_nor, valid_x_future_nor, valid_y_nor = apply_seq_norm(
                    x_valid_past, x_valid_future, y_valid_matrix, norm_dict)
                test_x_past_nor, test_x_future_nor, _ = apply_seq_norm(
                    x_test_past, x_test_future, None, norm_dict)
                ext_x_past_nor, ext_x_future_nor, _ = apply_seq_norm(
                    x_ext_past, x_ext_future, None, norm_dict)

                gc.collect()
                torch.cuda.empty_cache()

                seq_model = CustomLSTMTIME1(
                    zhixin=zhixin,
                    past_input_dim=train_x_past_nor.shape[-1],
                    future_input_dim=train_x_future_nor.shape[-1],
                    horizon=future_horizons_num,
                    hidden_dim=64,
                    num_layers=1,
                    lr=0.001,
                    batch_size=128,
                    num_epochs=60,
                    dropout=0.1,
                    patience=10
                )

                seq_model.train(
                    train_x_past_nor, train_x_future_nor, train_y_nor,
                    valid_x_past_nor, valid_x_future_nor, valid_y_nor
                )

                y_middle_matrix, upper_matrix, lower_matrix = seq_model.predict(
                    test_x_past_nor, test_x_future_nor,
                    norm_dict['y_mean'], norm_dict['y_std']
                )
                y_middle_external_matrix, upper_external_matrix, lower_external_matrix = seq_model.predict(
                    ext_x_past_nor, ext_x_future_nor,
                    norm_dict['y_mean'], norm_dict['y_std']
                )

                
                y_middle, upper, lower = flatten_pred_matrix(y_middle_matrix, upper_matrix, lower_matrix)
                y_middle_external, upper_external, lower_external = flatten_pred_matrix(
                    y_middle_external_matrix, upper_external_matrix, lower_external_matrix)

                y_true_eval = y_test_matrix.reshape(-1)
                y_true_ext_eval = y_ext_matrix.reshape(-1)

                # # Verify whether the test intervals used in the sequential forecasting task are exactly aligned
                # with those used in the original tabular forecasting task
                
                raw_test_target = build_nonoverlap_target(df_process_time, target_col, valid_end, test_end)
                raw_ext_target = build_nonoverlap_target(df_process_time, target_col, test_end, ext_end)

                if len(raw_test_target) >= len(y_true_eval):
                    assert np.allclose(raw_test_target[:len(y_true_eval)], y_true_eval), 'The test set of LSTMTIME is not aligned with the original test interval.'
                if len(raw_ext_target) >= len(y_true_ext_eval):
                    assert np.allclose(raw_ext_target[:len(y_true_ext_eval)], y_true_ext_eval), 'The external test set of LSTMTIME is not aligned with the original external test interval.'

                # 
                df_test_list = df_process.iloc[valid_end:test_end].copy()
                df_test_list_external = df_process.iloc[test_end:ext_end].copy()

                print(f'LSTMTIME test matrix shape: {y_middle_matrix.shape}')
                print(f'LSTMTIME external matrix shape: {y_middle_external_matrix.shape}')

            else:
                raise ValueError(f'Unsupported method: {method_choose}')


            true_y_list.append(y_true_eval)
            prdict_y_list.append(y_middle)
            lower_y_list.append(lower)
            up_y_list.append(upper)

            true_y_list_external.append(y_true_ext_eval)
            prdict_y_list_external.append(y_middle_external)
            lower_y_list_external.append(lower_external)
            up_y_list_external.append(upper_external)

            MAE, NRMSE, R2, MAPE, RMSE = evaluate_regress(y_middle, y_true_eval)
            point_predict.append([MAE, NRMSE, R2, MAPE, RMSE])

            MAE_ext, NRMSE_ext, R2_ext, MAPE_ext, RMSE_ext = evaluate_regress(y_middle_external, y_true_ext_eval)
            point_predict_external.append([MAE_ext, NRMSE_ext, R2_ext, MAPE_ext, RMSE_ext])

            quantile = 1 - (1 - zhixin1) / 2
            cacluate_interval = np.ones((5, len(quantile)))
            for i in range(len(quantile)):
                confidence_level = zhixin1[i]
                alpha = 1 - confidence_level
                lower_bounds = lower[:, i]
                upper_bounds = upper[:, i]

                picp_result, pinaw_result, cwc_result, ql_result, interval_score_result = cacluate_interval_score(
                    y_true_eval, lower_bounds, upper_bounds, 10, alpha
                )
                cacluate_interval[:, i] = [
                    picp_result, pinaw_result, cwc_result, ql_result, interval_score_result
                ]

            interval_predict.append(cacluate_interval)
            df_test_list_all.append(df_test_list)

            cacluate_interval_external = np.ones((5, len(quantile)))
            for i in range(len(quantile)):
                confidence_level = zhixin1[i]
                alpha = 1 - confidence_level
                lower_bounds = lower_external[:, i]
                upper_bounds = upper_external[:, i]

                picp_result, pinaw_result, cwc_result, ql_result, interval_score_result = cacluate_interval_score(
                    y_true_ext_eval, lower_bounds, upper_bounds, 10, alpha
                )
                cacluate_interval_external[:, i] = [
                    picp_result, pinaw_result, cwc_result, ql_result, interval_score_result
                ]

            interval_predict_external.append(cacluate_interval_external)

            output_df = converge(
                station_name_list,
                df_test_list,
                true_y_list,
                prdict_y_list,
                up_y_list,
                lower_y_list,
                zhixin1
            )

    true_y_list_all.append(true_y_list)
    prdict_y_list_all.append(prdict_y_list)
    lower_y_list_all.append(lower_y_list)
    up_y_list_all.append(up_y_list)
    output_df_ngboost_all.append(output_df)
    point_predict_all.append(point_predict)
    interval_predict_all.append(interval_predict)

    true_y_list_external_all.append(true_y_list_external)
    prdict_y_list_external_all.append(prdict_y_list_external)
    lower_y_list_external_all.append(lower_y_list_external)
    up_y_list_external_all.append(up_y_list_external)
    point_predict_external_all.append(point_predict_external)
    interval_predict_external_all.append(interval_predict_external)

    NRMSE_array = np.zeros(len(point_predict))
    for i in range(len(point_predict)):
        NRMSE_array[i] = point_predict[i][1]
    print('*******************************************')
    print(method_choose + '_NRMSE')
    print(NRMSE_array)

    interval_IS_array = np.zeros(len(point_predict))
    for i in range(len(interval_predict)):
        for j in range(0, len(zhixin1)):
            interval_IS_array[i] = interval_IS_array[i] + interval_predict[i][4, j]
    interval_IS_array1 = interval_IS_array / len(zhixin1)

    print('*******************************************')
    print(method_choose + '_IS')
    print(interval_IS_array1)


metrics_label_list = [
    'MAE', 'NRMSE', 'R2', 'MAPE', 'RMSE',
    'PICP', 'PINAW', 'CWC', 'QL', 'IS'
]

result2_df_last = build_metric_summary_df(
    metrics_label_list=metrics_label_list,
    point_predict_all=point_predict_all,
    interval_predict_all=interval_predict_all,
    station_name_list=station_name_list,
    method_choose_list1=method_choose_list1
)
print(result2_df_last)

data_test_name1 =  'testday_time_' + str(day_set) + '_interval_result1.xlsx'
result2_df_last.to_excel(data_test_name1, index=True)

result2_df_last_external = build_metric_summary_df(
    metrics_label_list=metrics_label_list,
    point_predict_all=point_predict_external_all,
    interval_predict_all=interval_predict_external_all,
    station_name_list=station_name_list,
    method_choose_list1=method_choose_list1
)

data_test_name2 =  'external_testday_time_' + str(day_set) + '_interval_result1.xlsx'
result2_df_last_external.to_excel(data_test_name2, index=True)

result2_df_last

d:\python learning\SolarUPE_project\datasets\data_NG
Processing TransformerTIME - data_NG - PV_NG_01
Epoch [001/060] Train Loss: 0.309614 | Valid Loss: 0.191525
Epoch [002/060] Train Loss: 0.178907 | Valid Loss: 0.129266
Epoch [003/060] Train Loss: 0.129041 | Valid Loss: 0.100624
Epoch [004/060] Train Loss: 0.105616 | Valid Loss: 0.089917
Epoch [005/060] Train Loss: 0.092930 | Valid Loss: 0.081199
Epoch [006/060] Train Loss: 0.083770 | Valid Loss: 0.077613
Epoch [007/060] Train Loss: 0.079157 | Valid Loss: 0.075303
Epoch [008/060] Train Loss: 0.075366 | Valid Loss: 0.078091
Epoch [009/060] Train Loss: 0.071074 | Valid Loss: 0.073994
Epoch [010/060] Train Loss: 0.067504 | Valid Loss: 0.069689
Epoch [011/060] Train Loss: 0.065149 | Valid Loss: 0.068769
Epoch [012/060] Train Loss: 0.063697 | Valid Loss: 0.069715
Epoch [013/060] Train Loss: 0.062215 | Valid Loss: 0.068236
Epoch [014/060] Train Loss: 0.060761 | Valid Loss: 0.068718
Epoch [015/060] Train Loss: 0.059020 | Valid Loss: 0.067513

Metric               MAE                                         NRMSE  \
Method   TransformerTIME  LSTMTIME      LSTM       MLP TransformerTIME   
PV_NG_01        2.579086  3.181604  1.924989  1.956030        0.083834   
PV_NG_02        4.960306  4.538059  2.978870  3.464631        0.100144   

Metric                                              R2            ...  \
Method    LSTMTIME      LSTM       MLP TransformerTIME  LSTMTIME  ...   
PV_NG_01  0.104848  0.087340  0.089710        0.898141  0.840676  ...   
PV_NG_02  0.085803  0.069939  0.079248        0.909677  0.933694  ...   

Metric         CWC                        QL                                   \
Method        LSTM       MLP TransformerTIME   LSTMTIME       LSTM        MLP   
PV_NG_01  0.119010  0.295218       10.536710  16.822051   8.764651  11.046308   
PV_NG_02  0.096224  0.100550       21.816026  19.111958  14.247678  16.679527   

Metric                IS                                
Method   TransformerTIME  LSTMTIME      LSTM       MLP  
PV_NG_01        0.563222  1.011387  0.597009  0.739984  
PV_NG_02        1.214446  0.914169  0.626906  0.791738  

[2 rows x 40 columns]